# Chapter 01 — AI fluency, prompting and output validation · Practice

**Goal:** practise the whole fluency loop by hand: choose a mode, write a brief, see each model property fail and get fixed, check outputs for unsupported numbers, missing points and bias, pick a review level, compare drafts with a rubric, diagnose weak output, and make fixes persist. The mini-project improves one recurring task (a weekly status update) step by step, through one interface that also runs on Claude and OpenAI.

- Theory: [README.md](README.md) (read section 1.x before running the matching section here)
- Estimated time: 90–120 minutes
- Estimated API cost: **$0** offline. The two optional live cells (`RUN_LIVE = True`) cost about $0.01–0.02 in total.

**How it works.** Every model call goes to `fake`, a small scripted stand-in that reacts to what your prompt contains, the way a real model reacts to missing context. Run the notebook top to bottom with a fresh kernel. Live cells are marked **(live, costs money)** and do nothing unless you set `RUN_LIVE = True` and have keys in `.env`.

In [ ]:
import os
import re
from collections import Counter
from itertools import count

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())                 # keys come from .env, never hardcode them
RUN_LIVE = False                           # set True to run the two live cells (costs money)
CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")

### Offline toolkit (run this once)

`FakeLLM(respond)` stands in for a model: `respond(system, prompt)` returns the reply text, and every call is recorded in `.calls`. Each section defines its own small `respond` function so you can see exactly which part of the prompt changes the answer. `NOTES` is the source document used throughout: the notes of a weekly project meeting.

In [ ]:
class FakeLLM:
    """A scripted stand-in for a chat model: respond(system, prompt) -> text."""

    def __init__(self, respond):
        self.respond = respond
        self.calls: list[dict] = []

    def __call__(self, system: str, prompt: str) -> str:
        self.calls.append({"system": system, "prompt": prompt})
        return self.respond(system, prompt)


NOTES = """Project Atlas weekly sync, 3 Nov 2026
- Migration: 18 of 24 tables moved. The last 6 wait for vendor API keys (owner: Priya, due 10 Nov).
- Budget: 41,200 EUR spent of 60,000 EUR.
- Risk: the vendor rate limit of 100 requests per minute may slow the final load.
- Decision: go-live moves from 14 Nov to 21 Nov.
- Action: Marco drafts the customer notice by 7 Nov."""

REQUIRED_POINTS = ["18 of 24", "41,200", "21 Nov", "Priya", "Marco"]
print(NOTES)

## 1.1 AI fluency and the 4D framework

Delegation, Description, Discernment, Diligence, used in three modes: **automation** (AI does a defined task), **augmentation** (you and AI work turn by turn), **agency** (you set AI up to act later, for you or others). Most failures are a mode mismatch. [Theory](README.md#11-ai-fluency-and-the-4d-framework)

The rule below is a rough first pass, not a law: a task that is fully defined up front is automation, a task you set up for future use is agency, and everything open-ended is augmentation.

In [ ]:
def mode_for(defined_up_front: bool, configures_future_use: bool) -> str:
    if configures_future_use:
        return "agency"          # needs performance description + testing
    if defined_up_front:
        return "automation"      # needs a precise brief + a quality check
    return "augmentation"        # needs process description + ongoing discernment


TASKS = {
    "Summarize the meeting notes into five bullets": (True, False),
    "Shape a pricing strategy for a new market": (False, False),
    "Set up a Project that answers staff HR questions": (True, True),
    "Turn 40 survey comments into themes with counts": (True, False),
}
for task, flags in TASKS.items():
    print(f"{mode_for(*flags):<13} {task}")

## 1.2 How LLMs behave: four properties behind every surprise

Each cell below shows one property producing a surprise, then the targeted fix. [Theory](README.md#12-how-llms-behave-four-properties-behind-every-surprise)

### Knowledge: the model only knows its training data

In [ ]:
POLICY_2026 = "Parental leave: 16 weeks fully paid, for births from 1 Sep 2026."


def policy_respond(system, prompt):
    if "<inputs>" in prompt:                         # a source was attached
        return "Per the attached policy: 16 weeks fully paid (births from 1 Sep 2026)."
    return "Parental leave is 12 weeks paid, as set in the 2024 policy."   # stale knowledge


fake = FakeLLM(policy_respond)
print("no source :", fake("", "How long is our parental leave?"))
print("with source:", fake("", f"How long is our parental leave?\n<inputs>\n{POLICY_2026}\n</inputs>"))

### Working memory: attention over a long input is uneven

This fake only "attends" to the first 120 words of a document, a cartoon of how detail late in a long input gets missed. The fix is to point at the section.

In [ ]:
FILLER = " ".join(["The group reports results in euros across all regions."] * 15)
REPORT = ("Risk 1: a large share of revenue is billed in US dollars. " + FILLER
          + " Note 27: a currency hedge expires in March 2027 and is not renewed.")


def long_doc_respond(system, prompt):
    doc = prompt.split("<inputs>")[-1]
    visible = " ".join(doc.split()[:120])            # cartoon of uneven attention
    return "; ".join(r for r in ["Risk 1", "Note 27"] if r in visible) or "no risks found"


fake = FakeLLM(long_doc_respond)
print("whole report :", fake("", f"List currency risks.\n<inputs>{REPORT}"))
note = REPORT[REPORT.index("Note 27"):]
print("point to note:", fake("", f"List currency risks in this note.\n<inputs>{note}"))
print(f"(the report is {len(REPORT.split())} words; the fake attends to 120)")

### Steerability: a leading question gets agreement

In [ ]:
def steer_respond(system, prompt):
    if prompt.lower().startswith("confirm"):
        return "Yes, go-live is still 14 Nov."        # agrees with the premise
    return "Per the notes, go-live moves from 14 Nov to 21 Nov."


fake = FakeLLM(steer_respond)
print("leading:", fake("", f"Confirm go-live is still 14 Nov.\n<inputs>\n{NOTES}\n</inputs>"))
print("neutral:", fake("", f"What is the go-live date in the notes?\n<inputs>\n{NOTES}\n</inputs>"))

### Next-token prediction: without a source, specifics are guesses

Ask the same question three times. Without a source, the figure changes between runs, which is a strong sign it was invented. With the source it is stable.

In [ ]:
_guess = count()


def growth_respond(system, prompt):
    if "<inputs>" in prompt:
        return "Revenue grew 9% year on year."
    return f"Revenue grew {[12, 9, 15][next(_guess) % 3]}% year on year."


fake = FakeLLM(growth_respond)
print("no source  :", [fake("", "How much did revenue grow?") for _ in range(3)])
print("with source:", [fake("", "How much did revenue grow?\n<inputs>Revenue +9%</inputs>") for _ in range(3)])

## 1.3 Delegation: choosing the mode of collaboration

Good candidates for AI: cheap to check, costly to write. Keep with a human: checking costs as much as doing, or the step cannot be undone. [Theory](README.md#13-delegation-choosing-the-mode-of-collaboration)

In [ ]:
def who_does(step: dict) -> str:
    if not step["reversible"] or step["needs_judgment"]:
        return "human keeps"
    if step["cheap_to_check"] and step["costly_to_write"]:
        return "AI drafts, human checks"
    return "together"


STATUS_UPDATE_STEPS = [
    {"step": "Draft the update from the notes", "reversible": True, "needs_judgment": False,
     "cheap_to_check": True, "costly_to_write": True},
    {"step": "List candidate risks from the notes", "reversible": True, "needs_judgment": False,
     "cheap_to_check": False, "costly_to_write": False},
    {"step": "Send it to the steering committee", "reversible": False, "needs_judgment": True,
     "cheap_to_check": True, "costly_to_write": False},
]
for s in STATUS_UPDATE_STEPS:
    print(f"{who_does(s):<24} {s['step']}")

## 1.4 Description: a repeatable prompt structure

The README's `gaps()` and `render()`. A bare one-line request misses most components; the full brief has them all. [Theory](README.md#14-description-a-repeatable-prompt-structure)

In [ ]:
COMPONENTS = ("task", "context", "inputs", "output",
              "constraints", "examples", "when_unsure")
REQUIRED = ("task", "context", "inputs", "output", "when_unsure")


def gaps(brief: dict[str, str]) -> list[str]:
    return [c for c in REQUIRED if not brief.get(c, "").strip()]


def render(brief: dict[str, str]) -> str:
    return "\n\n".join(f"<{c}>\n{brief[c]}\n</{c}>"
                       for c in COMPONENTS if brief.get(c, "").strip())


bare = {"task": "Write a status update for Project Atlas."}
full = {
    "task": "Write the weekly status update for Project Atlas.",
    "context": "For the steering committee, who decide on budget and dates.",
    "inputs": NOTES,
    "output": "Five bullets, under 20 words each, plain English.",
    "constraints": "Use only facts from the inputs. No new numbers.",
    "when_unsure": "Write 'not in the notes' instead of guessing.",
}
print("bare gaps:", gaps(bare))
print("full gaps:", gaps(full))
print("\n" + render(full)[:400], "...")

## 1.5 Adding useful context

Context belongs where its lifetime fits. This lookup is the README table; feature names change, so check each vendor's help pages. [Theory](README.md#15-adding-useful-context)

In [ ]:
PLACEMENT = {
    "this request": {"claude": "chat message, attached files", "chatgpt": "chat message, attached files"},
    "one piece of work": {"claude": "Project instructions + project knowledge",
                          "chatgpt": "Project instructions + files"},
    "everything you do": {"claude": "profile preferences, styles, memory",
                          "chatgpt": "custom instructions, memory"},
    "other people's chats": {"claude": "a shared Project", "chatgpt": "a custom GPT or shared project"},
}

CONTEXT_ITEMS = [
    ("Today's meeting notes", "this request"),
    ("The steering committee prefers five bullets", "one piece of work"),
    ("I write in British English", "everything you do"),
    ("How the support team should answer refund questions", "other people's chats"),
]
for item, lifetime in CONTEXT_ITEMS:
    where = PLACEMENT[lifetime]
    print(f"{item:<55} Claude: {where['claude']:<42} ChatGPT: {where['chatgpt']}")

## 1.6 Matching the prompt to the task type

Each task type needs one component more than the others. For brainstorming, **generate many, then judge in a separate step**: here the fake generates names and plain code shortlists them against the criteria. [Theory](README.md#16-matching-the-prompt-to-the-task-type)

In [ ]:
TYPE_NEEDS = {"analysis": "explicit criteria", "research": "source rules",
              "drafting": "audience", "brainstorming": "how many options"}


def brainstorm_respond(system, prompt):
    if "best" in prompt.lower():
        return "Smart Sync"                                   # one safe, average answer
    return "\n".join(["Smart Sync", "Tidewell", "Bridgeline", "Relay", "Sincronía Max",
                      "Northstar Data", "Atlas Link", "Hopper", "Mosaic", "SyncPro Ultimate"])


fake = FakeLLM(brainstorm_respond)
print("extra component per type:", TYPE_NEEDS)
print("one-shot 'best':", fake("", "What is the single best name for our sync feature?"))

names = fake("", "Give 10 varied names for our sync feature.").splitlines()
TAKEN = {"relay", "mosaic"}                                   # names your legal team says are taken


def shortlist(names, max_words=2, max_len=12):
    return [n for n in names if len(n.split()) <= max_words and len(n) <= max_len
            and n.lower() not in TAKEN and n.isascii()]


print("shortlist:", shortlist(names))

## 1.7 Discernment: accuracy, completeness, consistency and bias

`check_output` from the README, run on a weak draft (no source in the prompt) and a grounded draft. Then a regenerate-and-compare check and a swap test for bias. [Theory](README.md#17-discernment-accuracy-completeness-consistency-and-bias)

In [ ]:
NUM = re.compile(r"\d[\d,.]*\d|\d")


def numbers(text: str) -> set[str]:
    return {n.replace(",", "") for n in NUM.findall(text)}


def check_output(answer: str, source: str, required: list[str]) -> dict:
    return {
        "numbers_not_in_source": sorted(numbers(answer) - numbers(source)),
        "quotes_not_in_source": [q for q in re.findall(r'"([^"]+)"', answer)
                                 if q.lower() not in source.lower()],
        "missing_points": [r for r in required if r.lower() not in answer.lower()],
    }


WEAK = ('Project Atlas is about 90% complete and spending is on track at 45,000 EUR. '
        'Go-live remains 14 Nov. The vendor said "no blockers remain".')
GROUNDED = ("- Migration: 18 of 24 tables moved; 6 wait for vendor API keys (Priya, 10 Nov).\n"
            "- Budget: 41,200 EUR spent of 60,000 EUR.\n- Go-live moves to 21 Nov.\n"
            "- Marco drafts the customer notice by 7 Nov.")
print("weak    :", check_output(WEAK, NOTES, REQUIRED_POINTS))
print("grounded:", check_output(GROUNDED, NOTES, REQUIRED_POINTS))

> **Observe:** `14 Nov` is in the source, so the number check passes it, yet it is wrong: the notes say go-live *moved* to 21 Nov. Only the missing-points check (`21 Nov`) catches it. A clean number check does not mean the output is right; it only rules out one kind of error.

### Regenerate and compare

Claims that change between runs are weak. Reusing the no-source fake from 1.2:

In [ ]:
fake = FakeLLM(growth_respond)
runs = [fake("", "How much did revenue grow?") for _ in range(3)]
unstable = len({tuple(sorted(numbers(r))) for r in runs}) > 1
print(runs, "-> unstable, verify against a source" if unstable else "-> stable")

### Swap test for bias

Same prompt, one attribute changed. This fake has a built-in bias, the kind you would only see by comparing.

In [ ]:
NOTES_REF = "Led the data migration; cut load time by 30%; trained 4 new analysts."


def reference_respond(system, prompt):
    name = re.search(r"for (\w+)", prompt).group(1)
    words = "supportive and pleasant" if name in {"Anna", "Maria"} else "decisive and strong"
    return f"{name} is {words}. {NOTES_REF}"


def swap_test(llm, template: str, values: list[str]) -> dict[str, set[str]]:
    outs = {v: set(re.findall(r"[a-z]+", llm("", template.format(v)).lower())) - {v.lower()}
            for v in values}
    shared = set.intersection(*outs.values())
    return {v: words - shared for v, words in outs.items()}


fake = FakeLLM(reference_respond)
print(swap_test(fake, "Write a two-line reference for {} using these notes: " + NOTES_REF,
                ["Anna", "James"]))

## 1.8 When a human must verify

Decide the review level from the task, never from how confident the output sounds. [Theory](README.md#18-when-a-human-must-verify)

In [ ]:
def review_level(stakes: str, external: bool, reversible: bool, checkable: bool) -> str:
    if stakes == "high" or not checkable:
        return "expert_signoff"
    if external or not reversible or stakes == "medium":
        return "fact_check"
    return "self_check"


CASES = {
    "Brainstormed titles for an internal workshop": ("low", False, True, True),
    "Weekly status update to the steering committee": ("medium", False, True, True),
    "Customer email announcing the new go-live date": ("low", True, False, True),
    "Summary of contract liability sent to a client": ("high", True, False, True),
    "Answer about a tax rule you cannot check yourself": ("low", False, True, False),
}
for case, args in CASES.items():
    print(f"{review_level(*args):<15} {case}")

## 1.9 Editing, comparing and formatting for the audience

Write the rubric first, then score every draft against it. `readability` (average words per sentence) is a rough proxy for the length dial. [Theory](README.md#19-editing-comparing-and-formatting-for-the-audience)

**Format follows use:** inline text for a one-off answer, a document (Claude artifacts, ChatGPT canvas) for something you edit and share, a table for comparisons, CSV or JSON for anything another tool will read.

In [ ]:
def score(draft: str, must_include: list[str], max_words: int, banned: list[str]) -> int:
    text = draft.lower()
    points = sum(term.lower() in text for term in must_include)
    points += len(draft.split()) <= max_words
    points -= sum(word.lower() in text for word in banned)
    return points


def best(drafts: dict[str, str], **rubric) -> str:
    return max(drafts, key=lambda k: score(drafts[k], **rubric))


def readability(text: str) -> float:
    sentences = [s for s in re.split(r"[.!?\n]+", text) if s.strip()]
    return round(len(text.split()) / max(len(sentences), 1), 1)


RUBRIC = dict(must_include=["21 Nov", "41,200"], max_words=60, banned=["synerg", "leverag"])
DRAFTS = {
    "warm": "Great news from the team! Thanks to everyone, we keep leveraging synergies "
            "and the project is moving ahead nicely toward launch.",
    "complete": GROUNDED,
    "long": "This comprehensive update leverages input from all workstreams. " + GROUNDED * 2,
}
for k, d in DRAFTS.items():
    print(f"{k:<9} score={score(d, **RUBRIC):>2}  words/sentence={readability(d)}")
print("best:", best(DRAFTS, **RUBRIC))

## 1.10 The Description-Discernment loop: diagnosing weak output

Name the symptom, look up the root cause, change one thing. Corrections you keep typing get promoted into standing instructions. [Theory](README.md#110-the-description-discernment-loop-diagnosing-weak-output)

In [ ]:
DIAGNOSIS = {
    "generic":         ("under-specification", "add context, audience and purpose"),
    "invented facts":  ("missing source", "attach the source; 'use only'; allow 'not stated'"),
    "wrong format":    ("output not described", "state the format; 3-5 varied examples"),
    "half ignored":    ("too much in one request", "split into steps and check each"),
    "rule ignored":    ("buried or conflicting instructions", "one labelled rules block"),
    "outdated":        ("knowledge cutoff", "provide current sources or turn on search"),
    "late detail missed": ("working memory", "point to the section; split the document"),
}


def promote(corrections: list[str], threshold: int = 3) -> list[str]:
    counts = Counter(c.strip().lower() for c in corrections)
    return [c for c, n in counts.most_common() if n >= threshold]


for symptom in ["invented facts", "generic"]:
    cause, fix = DIAGNOSIS[symptom]
    print(f"{symptom:<15} -> {cause:<22} -> {fix}")

CORRECTION_LOG = ["List owners by name", "use bullets", "list owners by name ",
                  "Use fiscal quarters", "List owners by name"]
print("promote:", promote(CORRECTION_LOG))

## 1.11 Diligence: own it, disclose it, deploy it carefully

Right tool, right data, checked output, honest credit, as a pre-share checklist. [Theory](README.md#111-diligence-own-it-disclose-it-deploy-it-carefully)

In [ ]:
APPROVED_TOOLS = {"claude-enterprise", "chatgpt-enterprise"}
ALLOWED_DATA = {"claude-enterprise": {"public", "internal", "confidential"},
                "chatgpt-enterprise": {"public", "internal", "confidential"},
                "personal-free-account": {"public"}}


def diligence_problems(item: dict) -> list[str]:
    problems = []
    if item["tool"] not in APPROVED_TOOLS:
        problems.append("creation: tool not approved")
    if item["data_class"] not in ALLOWED_DATA.get(item["tool"], set()):
        problems.append(f"creation: {item['data_class']} data not allowed in {item['tool']}")
    if not item["verified"]:
        problems.append("deployment: not verified at the required review level")
    if item["audience_expects_disclosure"] and not item["disclosed"]:
        problems.append("transparency: AI use not disclosed")
    return problems


print(diligence_problems({"tool": "personal-free-account", "data_class": "confidential",
                          "verified": False, "audience_expects_disclosure": True,
                          "disclosed": False}))
print(diligence_problems({"tool": "claude-enterprise", "data_class": "internal",
                          "verified": True, "audience_expects_disclosure": False,
                          "disclosed": False}))

## 1.12 Claude and ChatGPT side by side, and in your role

One small interface, three backends: `fake` (offline), `claude` (Messages API) and `openai` (Responses API). The live functions are the README's; they are only called when `RUN_LIVE` is `True`. [Theory](README.md#112-claude-and-chatgpt-side-by-side-and-in-your-role)

In [ ]:
def ask_claude(system: str, prompt: str) -> str:
    import anthropic
    msg = anthropic.Anthropic().messages.create(
        model=CLAUDE_MODEL, max_tokens=2048, system=system,
        messages=[{"role": "user", "content": prompt}])
    return "".join(b.text for b in msg.content if b.type == "text")


def ask_openai(system: str, prompt: str) -> str:
    import openai
    resp = openai.OpenAI().responses.create(
        model=OPENAI_MODEL, instructions=system, input=prompt)
    return resp.output_text


def ask(provider: str, system: str, prompt: str, fake: FakeLLM | None = None) -> str:
    if provider == "fake":
        return fake(system, prompt)
    if not RUN_LIVE:
        raise RuntimeError("live call blocked: set RUN_LIVE = True (costs money)")
    return {"claude": ask_claude, "openai": ask_openai}[provider](system, prompt)

In [ ]:
# (live, costs money: ~$0.005) one tiny request to each vendor through the same interface
if RUN_LIVE:
    for provider, key in [("claude", "ANTHROPIC_API_KEY"), ("openai", "OPENAI_API_KEY")]:
        if os.getenv(key):
            print(provider, "->", ask(provider, "Answer in one sentence.",
                                      "What does 'discernment' mean when working with AI?"))
        else:
            print(f"{provider}: skipped, {key} not set")
else:
    print("RUN_LIVE is False: live cell skipped")

## Mini-project: improve a recurring task through the Description-Discernment loop

The task: the weekly status update from `NOTES`. You run four versions of the brief, check each output with `check_output` and the rubric, diagnose what is wrong, and change **one** thing per version. The fake below reacts to the brief the way a real model does: no source means invented figures, no audience means jargon and length, no owner rule means no owners.

In [ ]:
def status_respond(system, prompt):
    p = (system + "\n" + prompt).lower()
    grounded = "<inputs>" in p and "only" in p
    if not grounded:
        return WEAK
    intro = "" if "<context>" in p else (
        "This comprehensive update leverages input from all workstreams to give "
        "a holistic view of progress, risks and next steps across Project Atlas. ")
    owners = "name the owner" in p
    facts = [
        "Migration: 18 of 24 tables moved; 6 wait for vendor API keys"
        + (" (owner: Priya, due 10 Nov)" if owners else " (due 10 Nov)"),
        "Budget: 41,200 EUR spent of 60,000 EUR",
        "Risk: vendor rate limit (100 requests per minute) may slow the final load",
        "Decision: go-live moves from 14 Nov to 21 Nov",
        ("Marco drafts" if owners else "Draft") + " the customer notice by 7 Nov",
    ]
    if "bullet" in p:
        return intro + "\n".join(f"- {f}" for f in facts)
    return intro + ". ".join(facts) + "."


def evaluate(answer: str) -> dict:
    result = check_output(answer, NOTES, REQUIRED_POINTS)
    result["rubric"] = score(answer, must_include=REQUIRED_POINTS, max_words=70,
                             banned=["leverag", "holistic", "comprehensive"])
    result["words"] = len(answer.split())
    return result


fake = FakeLLM(status_respond)

In [ ]:
VERSIONS = {}

# v1: the bare request people usually type
VERSIONS["v1 bare"] = {"task": "Write the weekly status update for Project Atlas."}

# v2: diagnosis of v1 = "invented facts" -> attach the source and restrict to it
VERSIONS["v2 + source"] = dict(VERSIONS["v1 bare"], inputs=NOTES,
                               constraints="Use only facts from the inputs. No new numbers.",
                               when_unsure="Write 'not in the notes' instead of guessing.")

# v3: diagnosis of v2 = "generic" and "wrong format" -> audience and output
VERSIONS["v3 + audience"] = dict(VERSIONS["v2 + source"],
                                 context="For the steering committee, who decide on budget and dates.",
                                 output="Five bullets, under 20 words each, plain English.")

for name, brief in VERSIONS.items():
    answer = ask("fake", "", render(brief), fake=fake)
    print(f"== {name}  gaps={gaps(brief)}\n{answer}\n-> {evaluate(answer)}\n")

In [ ]:
# v3 is accurate and readable, but the owners (Priya, Marco) are missing. Over three weeks
# you would type the same correction again and again: promote it so it persists, then run v4.
weekly_corrections = ["List owners by name", "list owners by name", "Shorter please",
                      "List owners by name"]
standing = promote(weekly_corrections)
print("promoted:", standing)

VERSIONS["v4 + standing rule"] = dict(
    VERSIONS["v3 + audience"],
    constraints=VERSIONS["v3 + audience"]["constraints"] + " Name the owner of every action.")

print(f"\n{'version':<20}{'gaps':>5}{'invented':>10}{'missing':>9}{'rubric':>8}{'words':>7}")
for name, brief in VERSIONS.items():
    r = evaluate(ask("fake", "", render(brief), fake=fake))
    print(f"{name:<20}{len(gaps(brief)):>5}{len(r['numbers_not_in_source']):>10}"
          f"{len(r['missing_points']):>9}{r['rubric']:>8}{r['words']:>7}")
print("\nreview level for this task:", review_level("medium", False, True, True))

> **Observe:** each version changed one lever and fixed one diagnosed cause: v2 the invented figures, v3 the jargon and format, v4 the missing owners. Note that the rubric alone would mislead you: v1 scores higher than v2 only because it is short, although its figures are invented. Read all the columns together. The v4 brief is now your reusable template; in Claude or ChatGPT it belongs in the Project instructions, with `NOTES` attached each week.

**Try it:** delete `"only"` from the v2 constraints and rerun. The fake falls back to invented figures: without "use only the inputs" a model may mix the source with its own guesses.

**Last cell (optional, live):** the same brief on both vendors. Real models phrase things differently from the fake, so read `missing_points` critically: a point may be worded differently rather than absent. That judgment is Discernment. Next: [02_homework.ipynb](02_homework.ipynb).

In [ ]:
# (live, costs money: ~$0.01) run the final brief on both vendors and check each answer the same way
if RUN_LIVE:
    final_prompt = render(VERSIONS["v4 + standing rule"])
    for provider, key in [("claude", "ANTHROPIC_API_KEY"), ("openai", "OPENAI_API_KEY")]:
        if not os.getenv(key):
            print(f"{provider}: skipped, {key} not set")
            continue
        answer = ask(provider, "", final_prompt)
        print(f"== {provider}\n{answer}\n-> {evaluate(answer)}\n")
else:
    print("RUN_LIVE is False: live cell skipped")